# PlayMind | 01. Análisis exploratorio de datos (EDA)
**Proyecto final - Fundamentos de Inteligencia Artificial**  
**Entrega:** lunes 16 de noviembre de 2026 · **Equipo:** completar integrantes

Este cuaderno revisa el comportamiento de jugadores y las imágenes faciales (FER-2013) **sin entrenar modelos todavía**. Incluye distribuciones, correlaciones, valores atípicos, faltantes, conteo por emoción y balance de clases.

> **Antes de ejecutar:** coloca el CSV tabular y las carpetas de imágenes en `data/raw/`. Si todavía no están, el cuaderno explica qué falta; no inventa resultados. Se puede ejecutar desde JupyterLab abierto en la carpeta del repositorio.

## 1. Preparación: rutas y librerías
Si la terminal muestra `(Play_Mind)`, ya estás en tu entorno. Inicia Jupyter con `jupyter lab` desde la carpeta `PlayMind`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from IPython.display import display

sns.set_theme(style="whitegrid")

# Funciona si Jupyter se abrió en la raíz del repo o dentro de notebooks/.
actual = Path.cwd().resolve()
RAIZ = actual.parent if actual.name.lower() == 'notebooks' else actual
DATOS = RAIZ / 'data' / 'raw'
FIGURAS = RAIZ / 'reports' / 'figuras'
FIGURAS.mkdir(parents=True, exist_ok=True)
print('Proyecto:', RAIZ)
print('Carpeta esperada de datos:', DATOS)
print('Carpeta de gráficas:', FIGURAS)

## 2. Cargar datos tabulares
**Dataset propuesto:** *Predict Online Gaming Behavior Dataset* (Kaggle). El archivo suele llamarse `online_gaming_behavior_dataset.csv`.

Se escogerá un CSV cuyo nombre contenga `gaming`, `player` o `behavior` (y se excluirán archivos de FER). Si tienes otro nombre, edita manualmente `ruta_csv`.

In [ ]:
csvs = list(DATOS.rglob('*.csv')) if DATOS.exists() else []
candidatos = [p for p in csvs if any(s in p.name.lower() for s in ['gaming', 'player', 'behavior'])]
ruta_csv = sorted(candidatos)[0] if candidatos else None

if ruta_csv is None:
    datos = None
    print('FALTA EL CSV TABULAR. Descárgalo y guárdalo en data/raw/.')
    print('Fuente: https://www.kaggle.com/datasets/rabieelkharoua/predict-online-gaming-behavior-dataset')
    print('Archivos CSV encontrados:', [str(p.relative_to(RAIZ)) for p in csvs])
else:
    try:
        datos = pd.read_csv(ruta_csv)
    except UnicodeDecodeError:
        datos = pd.read_csv(ruta_csv, encoding='latin1')
    print('Archivo cargado:', ruta_csv.relative_to(RAIZ))
    print('Filas:', datos.shape[0], '| Columnas:', datos.shape[1])
    display(datos.head())

## 3. Dimensiones, tipos, duplicados y estadística descriptiva
Este primer vistazo sirve para conocer la cantidad de jugadores, columnas, tipos de variables y posibles inconsistencias.

In [ ]:
if datos is not None:
    print('Dimensiones:', datos.shape)
    print('Columnas y tipos:')
    display(pd.DataFrame({'columna': datos.columns, 'tipo': datos.dtypes.astype(str).values}))
    print('Filas duplicadas:', int(datos.duplicated().sum()))
    display(datos.describe(include='all').T)
else:
    print('Sin análisis tabular: primero agrega el archivo CSV.')

## 4. Valores faltantes
Calculamos cuántos datos faltan por variable. No se supondrá que están completos hasta ejecutar esta sección.

In [ ]:
if datos is not None:
    faltantes = pd.DataFrame({
        'faltantes': datos.isna().sum(),
        'porcentaje': (datos.isna().mean() * 100).round(2)
    }).sort_values('faltantes', ascending=False)
    display(faltantes)
    fig, ax = plt.subplots(figsize=(10, 4))
    faltantes['porcentaje'].plot.bar(ax=ax)
    ax.set_title('Porcentaje de datos faltantes por columna')
    ax.set_ylabel('Porcentaje (%)'); ax.set_xlabel('Variable')
    fig.tight_layout()
    fig.savefig(FIGURAS / '01_faltantes.png', dpi=150)
    plt.show()
    if faltantes['faltantes'].sum() == 0:
        print('Interpretación: en el CSV cargado no se detectaron valores faltantes.')
    else:
        print('Interpretación: revisar las columnas con faltantes antes del modelado.')

## 5. Distribuciones numéricas
Revisamos variables como horas jugadas, sesiones semanales, nivel del jugador y duración de sesión. Las formas de los histogramas pueden indicar concentración, sesgos o valores extremos.

In [ ]:
if datos is not None:
    preferidas = ['PlayTimeHours', 'SessionsPerWeek', 'PlayerLevel', 'AvgSessionDurationMinutes']
    columnas = [c for c in preferidas if c in datos.columns and pd.api.types.is_numeric_dtype(datos[c])]
    if not columnas:
        columnas = datos.select_dtypes(include=np.number).columns.drop('PlayerID', errors='ignore').tolist()[:4]
    if columnas:
        fig, ejes = plt.subplots(2, 2, figsize=(11, 7))
        for eje, col in zip(ejes.flat, columnas):
            eje.hist(datos[col].dropna(), bins=25)
            eje.set_title('Distribución de ' + col)
            eje.set_ylabel('Cantidad de jugadores')
        for eje in list(ejes.flat)[len(columnas):]: eje.axis('off')
        fig.tight_layout()
        fig.savefig(FIGURAS / '02_distribuciones.png', dpi=150)
        plt.show()
        display(datos[columnas].describe().T)
    else:
        print('No se encontraron columnas numéricas para graficar.')

## 6. Correlaciones
Usamos correlación de Pearson para explorar **relaciones lineales entre variables numéricas**. Una correlación no significa causalidad.

In [ ]:
if datos is not None:
    numericas = datos.select_dtypes(include=np.number).drop(columns=['PlayerID'], errors='ignore')
    if numericas.shape[1] >= 2:
        matriz = numericas.corr()
        display(matriz.round(2))
        fig, ax = plt.subplots(figsize=(9, 7))
        mapa = ax.imshow(matriz, cmap='coolwarm', vmin=-1, vmax=1)
        ax.set_xticks(range(len(matriz.columns)), matriz.columns, rotation=60, ha='right')
        ax.set_yticks(range(len(matriz.index)), matriz.index)
        for i in range(len(matriz.index)):
            for j in range(len(matriz.columns)):
                ax.text(j, i, f'{matriz.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
        fig.colorbar(mapa, ax=ax, shrink=0.8)
        ax.set_title('Correlaciones entre variables numéricas')
        fig.tight_layout()
        fig.savefig(FIGURAS / '03_correlaciones.png', dpi=150)
        plt.show()
    else:
        print('Se requieren al menos dos columnas numéricas para correlaciones.')

## 7. Valores atípicos (regla IQR)
Un valor atípico es una observación fuera del intervalo Q1 - 1.5 × IQR a Q3 + 1.5 × IQR. **No lo eliminaremos automáticamente**, porque puede representar un jugador real.

In [ ]:
if datos is not None:
    nums = datos.select_dtypes(include=np.number).drop(columns=['PlayerID'], errors='ignore')
    filas = []
    for col in nums.columns:
        serie = nums[col].dropna()
        if serie.empty: continue
        q1, q3 = serie.quantile([0.25, 0.75])
        iqr = q3-q1
        if iqr == 0:
            cantidad = 0  # Revisar por separado columnas con dispersión nula.
        else:
            cantidad = int(((serie < q1-1.5*iqr) | (serie > q3+1.5*iqr)).sum())
        filas.append({'variable':col, 'valores_atipicos':cantidad,
                      'porcentaje':round(100*cantidad/len(serie),2)})
    tabla_atipicos = pd.DataFrame(filas)
    display(tabla_atipicos)
    columnas_box = [c for c in ['PlayTimeHours', 'SessionsPerWeek', 'PlayerLevel'] if c in nums.columns]
    if columnas_box:
        fig, ejes = plt.subplots(1, len(columnas_box), figsize=(4*len(columnas_box), 4), squeeze=False)
        for eje,col in zip(ejes.flat,columnas_box):
            eje.boxplot(datos[col].dropna())
            eje.set_xticks([1], [col])
            eje.set_title('Atípicos en '+col)
        fig.tight_layout()
        fig.savefig(FIGURAS / '04_atipicos.png', dpi=150)
        plt.show()

## 8. Variable objetivo y balance del dataset tabular
Se propone **`EngagementLevel`** con las categorías **Low, Medium y High**, porque este dataset registra interés o participación, **no un abandono comprobado**. No llamaremos "abandono" al objetivo salvo que tengamos etiquetas reales de abandono.

In [ ]:
OBJETIVO = 'EngagementLevel'
if datos is not None:
    if OBJETIVO in datos.columns:
        conteos = datos[OBJETIVO].value_counts(dropna=False)
        display(pd.DataFrame({'jugadores':conteos, 'porcentaje':(conteos/len(datos)*100).round(2)}))
        fig, ax = plt.subplots(figsize=(6.5,4))
        conteos.plot.bar(ax=ax)
        ax.set_title('Distribución de niveles de participación')
        ax.set_xlabel('Nivel de participación'); ax.set_ylabel('Jugadores')
        fig.tight_layout()
        fig.savefig(FIGURAS / '05_objetivo_participacion.png', dpi=150)
        plt.show()
        columnas_grupo = [c for c in ['PlayTimeHours','SessionsPerWeek','PlayerLevel','AvgSessionDurationMinutes'] if c in datos.columns]
        if columnas_grupo:
            print('Promedio de variables por nivel de participación:')
            display(datos.groupby(OBJETIVO)[columnas_grupo].mean().round(2))
    else:
        print('No aparece EngagementLevel; verifiquen el CSV y consulten al docente si se usará otra etiqueta.')

## 9. Revisar imágenes FER-2013 por clase
**Ubicación esperada** al descomprimir el dataset de Kaggle:

```text
data/raw/fer2013/train/angry/*.jpg
data/raw/fer2013/train/happy/*.jpg
...
data/raw/fer2013/test/neutral/*.jpg
```

Detectamos automáticamente carpetas `train`, `test` o `val`, aunque estén dentro de una carpeta adicional. Para el prototipo seleccionamos **cinco emociones**: enojado, feliz, neutral, triste y sorprendido.

In [ ]:
CLASES = {'angry':'Enojado', 'happy':'Feliz', 'neutral':'Neutral', 'sad':'Triste', 'surprise':'Sorprendido'}
EXTS = {'.jpg','.jpeg','.png','.bmp'}
filas_img = []
archivos_por_clase = {}
if DATOS.exists():
    splits = [p for p in DATOS.rglob('*') if p.is_dir() and p.name.lower() in {'train','test','val','validation'}]
    for carpeta_split in splits:
        for carpeta_clase in carpeta_split.iterdir():
            if not carpeta_clase.is_dir(): continue
            clave = carpeta_clase.name.lower()
            if clave in CLASES:
                ejemplos = [p for p in carpeta_clase.iterdir() if p.is_file() and p.suffix.lower() in EXTS]
                if not ejemplos: continue
                split = carpeta_split.name.lower()
                filas_img.append({'split':split,'clase':CLASES[clave],'imagenes':len(ejemplos)})
                archivos_por_clase.setdefault(CLASES[clave],[]).extend(ejemplos)

tabla_imagenes = pd.DataFrame(filas_img)
if tabla_imagenes.empty:
    print('Aún no hay carpetas de imágenes FER-2013 en data/raw/.')
    print('Descarga y descomprime: https://www.kaggle.com/datasets/msambare/fer2013')
else:
    print('Conteo de imágenes por conjunto y emoción:')
    display(tabla_imagenes.pivot_table(index='clase', columns='split',values='imagenes',aggfunc='sum',fill_value=0))
    print('Total de imágenes en las cinco emociones:', int(tabla_imagenes['imagenes'].sum()))

## 10. ¿Hay desbalance entre las emociones?
Graficamos el número de fotografías por emoción, y comparamos la clase con más ejemplos contra la de menos ejemplos. La diferencia puede requerir ponderar las clases durante el entrenamiento.

In [ ]:
if not tabla_imagenes.empty:
    conteo_emociones = tabla_imagenes.groupby('clase')['imagenes'].sum().sort_values(ascending=False)
    display(pd.DataFrame({'imagenes':conteo_emociones,'porcentaje':(conteo_emociones/conteo_emociones.sum()*100).round(2)}))
    fig, ax = plt.subplots(figsize=(8,4))
    conteo_emociones.plot.bar(ax=ax)
    ax.set_title('Cantidad de imágenes FER-2013 por emoción (5 clases)')
    ax.set_xlabel('Emoción'); ax.set_ylabel('Número de imágenes')
    fig.tight_layout()
    fig.savefig(FIGURAS / '06_balance_emociones.png', dpi=150)
    plt.show()
    razon = conteo_emociones.max()/conteo_emociones.min()
    print(f'Proporción clase mayor / clase menor: {razon:.2f}')
    print('Interpretación:', 'hay diferencia notable entre clases; revisar ponderación o aumento de datos.' if razon >= 1.5 else 'las clases están relativamente cercanas en tamaño.')

## 11. Ejemplos de fotografías por emoción
Se mostrará un ejemplo real de cada clase disponible. **No se crean imágenes artificiales** en este análisis.

In [ ]:
if archivos_por_clase:
    clases_disponibles = [c for c in CLASES.values() if c in archivos_por_clase]
    fig, ejes = plt.subplots(1, len(clases_disponibles), figsize=(2.7*len(clases_disponibles),3), squeeze=False)
    for eje, clase in zip(ejes.flat, clases_disponibles):
        archivo = sorted(archivos_por_clase[clase])[0]
        with Image.open(archivo) as img:
            eje.imshow(img.convert('L'), cmap='gray')
        eje.set_title(clase); eje.axis('off')
    fig.suptitle('Ejemplo por clase - FER-2013')
    fig.tight_layout()
    fig.savefig(FIGURAS / '07_ejemplos_emociones.png', dpi=150)
    plt.show()

## 12. Interpretación y acuerdos del equipo
Completar después de ejecutar con **datos reales** (no copiar resultados de otras páginas).

1. **Distribuciones:** ¿Qué variable está más concentrada? ¿Cuál tiene valores altos poco frecuentes?
2. **Faltantes:** ¿Cuáles columnas contienen registros vacíos? ¿Qué tratamiento proponen?
3. **Correlaciones:** ¿Qué pares de variables tienen relaciones lineales más intensas?
4. **Atípicos:** ¿Los extremos son errores o jugadores con comportamientos excepcionales?
5. **Participación:** ¿Las categorías `Low`, `Medium` y `High` están equilibradas?
6. **Imágenes:** ¿Cuál emoción tiene más ejemplos y cuál menos? ¿Hay desbalance?

**Decisión propuesta:** usar `EngagementLevel` como variable objetivo; considerar el abandono **solo como hipótesis de riesgo**, porque los datos tabulares no ofrecen una etiqueta real de abandono.

## 13. Objetivos y métricas para las siguientes semanas

| Módulo | Qué se quiere hacer | Cómo evaluarlo |
|---|---|---|
| Supervisado | Clasificar `EngagementLevel` | F1 macro, AUC multiclase OvR y matriz de confusión; división 80/20, CV de 5 particiones |
| No supervisado | Segmentar estilos de juego | Codo (inercia) y coeficiente de silueta; PCA para visualizar |
| CNN | Reconocer 5 emociones faciales | Accuracy, F1 macro y matriz de confusión |
| Refuerzo | Aprender a resolver FrozenLake 4×4 | Tasa de éxito en 100 episodios y recompensa media (con/sin deslizamiento) |

> **Precaución:** AUC multiclase requiere probabilidades o puntuaciones y se reporta cuando estén entrenados los modelos; no calculamos aquí ninguna métrica predictiva.

## 14. Evidencias y GitHub
El cuaderno guarda imágenes en `reports/figuras/`. Antes de entregar, agregar interpretación a cada gráfica y ejecutarlo de principio a fin con los datasets instalados.

Desde **Anaconda Prompt**, dentro de la carpeta local `PlayMind`:

```bash
git pull origin main
git add notebooks/01_eda.ipynb reports/figuras/
git commit -m "EDA inicial de jugadores y emociones"
git push origin main
```

Si trabajas en una rama del equipo, cambia `main` por tu rama y abre una solicitud de incorporación (*pull request*).

### Fuentes de los datos
- Tabular: https://www.kaggle.com/datasets/rabieelkharoua/predict-online-gaming-behavior-dataset
- Emociones: https://www.kaggle.com/datasets/msambare/fer2013
- Instrucciones internas: Proyecto Final, Versión 7, PlayMind, apartados 5 a 7.

**Registrar autor, fecha de descarga y licencia de cada fuente en `data/README.md`.**